# 🧹 Exercise 02 — Explore & Clean

**Day 1 · Guided**

Profile a DataFrame, fix dtypes, handle missing values, remove duplicates, and save a clean subset.

---

In [7]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# TODO: Replace with your city slug (same as Exercise 01)
CITY_SLUG = "barcelona"   # <-- change this if you use a different city

data_dir = Path("../data")
gpkg_candidates = [
    data_dir / f"raw_{CITY_SLUG}.gpkg",
    data_dir / f"{CITY_SLUG}_amenities_raw.gpkg",
]
csv_candidates = [
    data_dir / f"clean_{CITY_SLUG}.csv",
]

gpkg_path = next((path for path in gpkg_candidates if path.exists()), None)
csv_path = next((path for path in csv_candidates if path.exists()), None)

if gpkg_path is not None:
    gdf = gpd.read_file(gpkg_path)

    # Extract lat/lon from geometry and convert to a regular DataFrame
    gdf["lat"] = gdf.geometry.centroid.y
    gdf["lon"] = gdf.geometry.centroid.x
    df = pd.DataFrame(gdf.drop(columns=["geometry"]))
elif csv_path is not None:
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(
        f"Could not find raw data for {CITY_SLUG}. "
        f"Tried: {[str(path) for path in gpkg_candidates + csv_candidates]}"
    )

# Rename osmid → id, drop osmnx metadata
if "osmid" in df.columns:
    df = df.rename(columns={"osmid": "id"})
for col in ["nodes", "ways"]:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Loaded data for slug: {CITY_SLUG}")
print(f"Source: {gpkg_path or csv_path}")

Loaded 939 rows, 197 columns
Loaded data for slug: barcelona
Source: ..\data\barcelona_amenities_raw.gpkg


C:\Users\etmaglari\AppData\Local\Temp\ipykernel_91388\126483850.py:24: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y
C:\Users\etmaglari\AppData\Local\Temp\ipykernel_91388\126483850.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.geometry.centroid.x


### 2.1 📊 Profile the data

Before fixing anything, understand what you have. Three best friends: `.info()`, `.describe()`, `.dtypes`.

In [8]:
# .info() gives you: columns, non-null counts, dtypes, memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 939 entries, 0 to 938
Columns: 197 entries, element to lon
dtypes: float64(2), int64(1), str(194)
memory usage: 1.5 MB


In [9]:
# TODO: Run .describe() — what does it tell you about lat/lon?
# Does the range make sense for your city?

In [10]:
# TODO: Check the dtypes of lat and lon.
# Are they float64? If not, that's a problem we need to fix.
# Hint: df["lat"].dtype

**Think:** How many columns? What % non-null? Are `lat`/`lon` float64? Does the coordinate range match your district?

### 2.2 🔧 Fix data types

Coordinates must be numeric. In some exports they arrive as strings.

In [11]:
# Ensure lat/lon are float — pd.to_numeric handles strings → float
# errors="coerce" turns unparseable values into NaN instead of crashing
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

print(f"lat dtype: {df['lat'].dtype}")
print(f"lon dtype: {df['lon'].dtype}")

# Drop rows where conversion failed
before = len(df)
df = df.dropna(subset=["lat", "lon"])
print(f"Dropped {before - len(df)} rows with invalid coordinates")

lat dtype: float64
lon dtype: float64
Dropped 0 rows with invalid coordinates


### 2.3 🕳️ Handle missing values

Most tag columns are >90% empty — normal for OSM, not every node has every tag.

In [12]:
# Calculate fill rate for each column
fill_rate = (df.notna().sum() / len(df) * 100).sort_values(ascending=False)

print("Top 20 columns by fill rate:")
print(fill_rate.head(20).round(1).to_string())

print(f"\nColumns with <1% fill rate: {(fill_rate < 1).sum()}")
print(f"Columns with <5% fill rate: {(fill_rate < 5).sum()}")

Top 20 columns by fill rate:
element             100.0
id                  100.0
amenity             100.0
lon                 100.0
lat                 100.0
name                 95.5
addr:street          70.5
addr:housenumber     61.9
check_date           47.0
wheelchair           33.1
cuisine              29.9
addr:postcode        29.5
name:ca              25.9
addr:city            24.5
opening_hours        23.9
website              15.9
phone                15.8
ref                  13.1
outdoor_seating      12.6
source               10.5

Columns with <1% fill rate: 126
Columns with <5% fill rate: 161


In [13]:
# TODO: Drop columns with very low fill rate (<1%)
# But keep important columns even if sparse: cuisine, opening_hours, phone, website, wheelchair

# Hint:
# sparse_cols = fill_rate[fill_rate < 1].index.tolist()
# keep_anyway = {"cuisine", "opening_hours", "phone", "website", "wheelchair"}
# drop_cols = [c for c in sparse_cols if c not in keep_anyway]
# df = df.drop(columns=drop_cols)

### 2.4 🔁 Remove duplicates

Same coordinates but different IDs — happens from mass imports or editing mistakes.

In [14]:
# TODO: How many duplicate coordinates are there?
# Hint: df.duplicated(subset=["lat", "lon"], keep=False).sum()

In [15]:
# TODO: Look at some duplicates — are they truly the same place or different things at the same spot?
# Hint:
# dupes = df[df.duplicated(subset=["lat", "lon"], keep=False)]
# dupes.sort_values(["lat", "lon"]).head(10)[["id", "lat", "lon", "amenity", "name"]]

In [16]:
# TODO: Remove duplicates (keep first occurrence)
# Hint:
# before = len(df)
# df = df.drop_duplicates(subset=["lat", "lon"], keep="first")
# print(f"Removed {before - len(df)} duplicates")

**Think:** We used `keep="first"` — is that always right? What if the second duplicate has better tag data?

### 2.5 🏷️ Extract a primary name

OSM has `name`, `name:en`, `name:sv`, `name:ar`… We need one usable column.

In [17]:
# What name columns do we have?
name_cols = [c for c in df.columns if c.startswith("name")]
print(f"Name-related columns: {sorted(name_cols)}")

Name-related columns: ['name', 'name:2011-2016', 'name:ca', 'name:en', 'name:es', 'name:et', 'name:etymology:wikidata', 'name:fr', 'name:it', 'name:ko', 'name:ru']


In [18]:
# TODO: Create a 'primary_name' column.
# Priority: name > name:en > first available name:* column
# If no name is available, leave it as None.

# Hint: use df.apply() with a function that checks each column in order

### 2.6 💾 Select core columns & save

In [19]:
# TODO: Create a clean DataFrame with these core columns.
# Keep: id, lat, lon, amenity, primary_name, cuisine, opening_hours
# Also keep: phone, website, wheelchair, addr:street, addr:housenumber (if they exist)

# Hint:
# core_cols = ["id", "lat", "lon", "amenity", "primary_name", "cuisine", "opening_hours"]
# optional_cols = ["phone", "website", "wheelchair", "addr:street", "addr:housenumber"]
# available = [c for c in optional_cols if c in df.columns]
# df_clean = df[core_cols + available].copy()

In [20]:
# TODO: Save your clean DataFrame
# Hint: df_clean.to_csv(f"../data/clean_{CITY_SLUG}.csv", index=False)

---

✅ Correct dtypes · ✅ No duplicate coordinates · ✅ Sparse columns removed · ✅ `primary_name` column · ✅ Clean CSV saved

#### Checklist
- [ ] `df_clean.dtypes` → `lat`/`lon` are `float64`
- [ ] `df_clean.duplicated(subset=['lat', 'lon']).sum()` → `0`
- [ ] `df_clean['primary_name'].notna().mean()` → > 0.5

**Next →** [Exercise 03 — Tag Normalization](03_tag_normalization.ipynb)